# Full Initialization Workflow - Single `.mo` File

This notebook initializes one dynamic Modelica model from a single `.mo` file.

It takes the input model from `models/`, creates the auxiliary model, uses it to compute the initialization values, and generates the initialized model in `outputs/`.

In [ ]:
import logging
import os
import sys

sys.path.append(os.path.abspath(".."))

logging.getLogger("OMPython").setLevel(logging.ERROR)

from OMPython import ModelicaSystemOMC, OMCSessionLocal

from scripts.helpers import *

In [ ]:
# User configuration: edit the values below for your model.

# Name of the dynamic model to initialize.
# The corresponding .mo file must be available in the models folder.
MODEL = "Demo_BESS"

# Modelica Standard Library.
OMLIB_DIR = os.path.join(os.path.expanduser("~"), ".openmodelica", "libraries")
os.environ["OPENMODELICALIBRARY"] = OMLIB_DIR
MODELICA_PKG_PATH = os.path.join(OMLIB_DIR, "Modelica 3.2.3+maint.om", "package.mo")

# Dynawo Modelica library, for now the one the Julia family downloads.
DYNAWO_PKG_PATH = os.path.abspath(
    os.path.join("..", "..", "julia_openmodelica", "dynawo_library", "Dynawo", "package.mo")
)

# Optional settings

# INIT model selection for components with multiple INIT profiles
INIT_MODEL_BY_COMPONENT = {
    # "generatorSynchronous": "GeneratorSynchronousInt_INIT",
}

# Leave empty to disable slack-specific handling.
SLACK_COMPONENT = ""

In [ ]:
# Internal paths derived from the user configuration above.
MODEL_DIR = os.path.abspath("models")
OUTPUT_DIR = os.path.abspath("outputs")

MODEL_FILE_PATH = os.path.join(MODEL_DIR, MODEL + ".mo")

AUX_MODEL = MODEL + "_auxiliary"
AUX_FILE_PATH = os.path.join(OUTPUT_DIR, AUX_MODEL + ".mo")

INITIALIZED_MODEL = MODEL + "_initialized"
INITIALIZED_FILE_PATH = os.path.join(OUTPUT_DIR, INITIALIZED_MODEL + ".mo")

if not os.path.isfile(MODEL_FILE_PATH):
    raise RuntimeError(f"Model file not found: {MODEL_FILE_PATH}")
if not os.path.isfile(DYNAWO_PKG_PATH):
    raise RuntimeError(f"Dynawo package not found: {DYNAWO_PKG_PATH}")
if not os.path.isfile(MODELICA_PKG_PATH):
    raise RuntimeError(f"Modelica package not found: {MODELICA_PKG_PATH}")
if not os.path.isdir(OUTPUT_DIR):
    raise RuntimeError(f"Output directory not found: {OUTPUT_DIR}")

## Build Auxiliary Model

The original dynamic model is loaded and checked before any changes are made. An auxiliary model is then generated and saved for the initialization simulation.

In [ ]:
DynamicOMC = OMCSessionLocal()
omc_call(DynamicOMC, f'loadFile("{MODELICA_PKG_PATH}")')
omc_call(DynamicOMC, "loadModel(Complex)")
omc_call(DynamicOMC, "loadModel(ModelicaServices)")
omc_call(DynamicOMC, f'loadFile("{DYNAWO_PKG_PATH}")')
omc_call(DynamicOMC, f'loadFile("{MODEL_FILE_PATH}")')

check_user_configuration_single(
    DynamicOMC,
    model=MODEL,
    slack_component=SLACK_COMPONENT,
    init_model_by_component=INIT_MODEL_BY_COMPONENT,
)

In [ ]:
source_components = get_all_components(DynamicOMC, MODEL)

# Drop any class left over from a previous run. A missing class is fine here.
send_expression(DynamicOMC, f"deleteClass({AUX_MODEL})")
print(f"Creating auxiliary model {AUX_MODEL} from {MODEL}")
omc_call(DynamicOMC, f'copyClass({MODEL}, "{AUX_MODEL}")')

apply_replacements(
    DynamicOMC,
    MODEL,
    AUX_MODEL,
    source_components,
    SLACK_COMPONENT,
)

delete_connections(DynamicOMC, AUX_MODEL, source_components)
delete_components(DynamicOMC, AUX_MODEL, source_components)

add_init_models(
    DynamicOMC,
    MODEL,
    AUX_MODEL,
    source_components,
    INIT_MODEL_BY_COMPONENT,
    SLACK_COMPONENT,
)

apply_LF_modifiers(
    DynamicOMC,
    MODEL,
    AUX_MODEL,
    source_components,
)

add_init_equations(
    DynamicOMC,
    MODEL,
    AUX_MODEL,
    source_components,
    INIT_MODEL_BY_COMPONENT,
    SLACK_COMPONENT,
)

clean_aux_equations(DynamicOMC, AUX_MODEL, source_components, SLACK_COMPONENT)
omc_call(DynamicOMC, f'saveModel("{AUX_FILE_PATH}", {AUX_MODEL})')

print("Wrote auxiliary model:", AUX_FILE_PATH)

## Extract Initialization Values

The written auxiliary model is loaded independently, checked, and simulated. Its simulation outputs provide the initialization values to apply to the original dynamic model.

In [ ]:
AuxiliaryOMC = OMCSessionLocal()
omc_call(AuxiliaryOMC, "loadModel(Complex)")
omc_call(AuxiliaryOMC, "loadModel(ModelicaServices)")
omc_call(AuxiliaryOMC, f'loadFile("{MODELICA_PKG_PATH}")')
omc_call(AuxiliaryOMC, f'loadFile("{DYNAWO_PKG_PATH}")')
omc_call(AuxiliaryOMC, f'loadFile("{AUX_FILE_PATH}")')

omc_call(AuxiliaryOMC, f"checkModel({AUX_MODEL})", parsed=False)
print("Auxiliary model checked successfully.")

In [ ]:
AuxiliarySystem = ModelicaSystemOMC(session=AuxiliaryOMC)
AuxiliarySystem.model(model_name=AUX_MODEL, model_file=AUX_FILE_PATH)
AuxiliarySystem.simulate(resultfile=AUX_MODEL + "_res.mat")

initializable_components = get_initializable_components(source_components, INIT_MODEL_BY_COMPONENT)

init_values_by_component = extract_all_initialization_values(
    AuxiliarySystem,
    source_components,
    INIT_MODEL_BY_COMPONENT,
)

initialized_component_names = sorted(init_values_by_component)
print("Extracted initialization values for:", ", ".join(initialized_component_names))

## Build Initialized Model

A copy of the validated original dynamic model is populated with the values extracted from the auxiliary simulation and saved as the initialized model.

In [ ]:
# Drop any class left over from a previous run. A missing class is fine here.
send_expression(DynamicOMC, f"deleteClass({INITIALIZED_MODEL})")
print(f"Creating initialized model {INITIALIZED_MODEL} from {MODEL}")
omc_call(DynamicOMC, f'copyClass({MODEL}, "{INITIALIZED_MODEL}")')

apply_initialization_modifiers(
    DynamicOMC,
    INITIALIZED_MODEL,
    initializable_components,
    init_values_by_component,
    INIT_MODEL_BY_COMPONENT,
)

omc_call(DynamicOMC, f'saveModel("{INITIALIZED_FILE_PATH}", {INITIALIZED_MODEL})')
print("Wrote initialized model:", INITIALIZED_FILE_PATH)

## Validate Initialized Model

The generated initialized model is loaded independently and simulated to verify that the final output can run.

In [ ]:
InitializedOMC = OMCSessionLocal()
omc_call(InitializedOMC, f'loadFile("{MODELICA_PKG_PATH}")')
omc_call(InitializedOMC, "loadModel(Complex)")
omc_call(InitializedOMC, "loadModel(ModelicaServices)")
omc_call(InitializedOMC, f'loadFile("{DYNAWO_PKG_PATH}")')

InitializedSystem = ModelicaSystemOMC(session=InitializedOMC)
InitializedSystem.model(model_name=INITIALIZED_MODEL, model_file=INITIALIZED_FILE_PATH)

omc_call(InitializedOMC, f"checkModel({INITIALIZED_MODEL})", parsed=False)
print("Initialized model checked successfully.")

initialized_resultfile_prefix = INITIALIZED_MODEL
simflags = simulation_flags_without_log_stats(InitializedOMC, INITIALIZED_MODEL)

send_expression(
    InitializedOMC,
    f'simulate({INITIALIZED_MODEL}, outputFormat="csv", '
    f'fileNamePrefix="{initialized_resultfile_prefix}", simflags="{simflags}")',
    parsed=False,
)

initialized_resultfile = os.path.join(
    str(InitializedSystem.getWorkDirectory()),
    initialized_resultfile_prefix + "_res.csv",
)
print("Initialized model simulated successfully.")

## Optional: Plot Simulation Result

The final cell is an optional example of visualizing a variable from the initialized-model simulation result.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Select a variable to visualize from the initialized simulation.
PLOT_VARIABLE = "BESS.terminal.V.im"

initialized_df = pd.read_csv(initialized_resultfile)
if PLOT_VARIABLE not in initialized_df.columns:
    raise RuntimeError(f'PLOT_VARIABLE "{PLOT_VARIABLE}" is not a variable in the simulation result.')

fig, ax = plt.subplots()
ax.plot(initialized_df["time"], initialized_df[PLOT_VARIABLE], label=PLOT_VARIABLE)
ax.legend(loc="lower right")
ax.set_title("Initialized dynamic model response")
ax.set_xlabel("Time (s)")
ax.set_ylabel(PLOT_VARIABLE)